# 08 Compare WLASL100 Models

This notebook compares the WLASL100 model experiments completed so far:

1. Baseline BiLSTM
2. Improved BiGRU + Temporal Attention
3. Small Transformer Encoder

The goal is to decide which architecture should be scaled to WLASL300.

Based on the current WLASL100 experiments, the main comparison metrics are:

- Best validation Macro F1
- Best validation Top-5 accuracy
- Test Top-1 accuracy
- Test Top-3 accuracy
- Test Top-5 accuracy
- Test Macro F1
- Generalisation behaviour
- Suitability for scaling to WLASL300 / WLASL1000 / WLASL2000

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

MODEL_DIR = PROJECT_ROOT / "models" / "ASL"
REPORT_DIR = PROJECT_ROOT / "reports" / "phase1_wlasl100"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

BIGRU_METRICS_FILE = REPORT_DIR / "improved_wlasl100_overall_metrics.csv"
TRANSFORMER_RESULT_FILE = MODEL_DIR / "transformer_encoder_wlasl100_result_summary.csv"
BASELINE_HISTORY_FILE = MODEL_DIR / "baseline_bilstm_wlasl100_history.csv"

COMPARISON_FILE = REPORT_DIR / "wlasl100_model_comparison.csv"
DECISION_FILE = REPORT_DIR / "wlasl100_model_selection_decision.md"

print("BiGRU metrics file exists:", BIGRU_METRICS_FILE.exists())
print("Transformer result file exists:", TRANSFORMER_RESULT_FILE.exists())
print("Baseline history file exists:", BASELINE_HISTORY_FILE.exists())
print("Comparison output:", COMPARISON_FILE)

## 1. Load model results

This section loads available result CSVs. If a result file is missing, it uses the known experiment results entered manually.

The baseline BiLSTM had very low validation performance and is included mainly as a reference point to show how much the improved model changed the project direction.

In [ ]:
result_rows = []

# Baseline BiLSTM reference result
# The original baseline was mainly used to prove the pipeline worked.
# It did not include a full saved result summary like later models.
baseline_row = {
    "dataset": "WLASL100",
    "model": "Baseline BiLSTM",
    "clean_samples": 1013,
    "classes": 100,
    "input_shape": "(60, 258)",
    "velocity_features": False,
    "best_val_f1": 0.0115,
    "best_val_top5": np.nan,
    "checkpoint_epoch": 17,
    "test_top1_accuracy": np.nan,
    "test_top3_accuracy": np.nan,
    "test_top5_accuracy": np.nan,
    "test_macro_f1": np.nan,
    "notes": "Initial baseline. Very low performance; used to validate the end-to-end pipeline."
}

result_rows.append(baseline_row)

# Improved BiGRU + Attention result
if BIGRU_METRICS_FILE.exists():
    bigru_df = pd.read_csv(BIGRU_METRICS_FILE)
    row = bigru_df.iloc[0].to_dict()

    bigru_row = {
        "dataset": row.get("dataset", "WLASL100"),
        "model": row.get("model", "BiGRU + Temporal Attention"),
        "clean_samples": int(row.get("clean_samples", 1013)),
        "classes": int(row.get("classes", 100)),
        "input_shape": "(60, 516)",
        "velocity_features": True,
        "best_val_f1": float(row.get("best_val_f1", 0.4723)),
        "best_val_top5": float(row.get("best_val_top5", 0.8221)),
        "checkpoint_epoch": int(row.get("checkpoint_epoch", 45)),
        "test_top1_accuracy": float(row.get("test_top1_accuracy", 0.4304)),
        "test_top3_accuracy": float(row.get("test_top3_accuracy", 0.7089)),
        "test_top5_accuracy": float(row.get("test_top5_accuracy", 0.7911)),
        "test_macro_f1": float(row.get("test_macro_f1", 0.3855)),
        "notes": "Best WLASL100 model so far. Strong Top-5 and stable improvement over baseline."
    }
else:
    bigru_row = {
        "dataset": "WLASL100",
        "model": "BiGRU + Temporal Attention",
        "clean_samples": 1013,
        "classes": 100,
        "input_shape": "(60, 516)",
        "velocity_features": True,
        "best_val_f1": 0.4723,
        "best_val_top5": 0.8221,
        "checkpoint_epoch": 45,
        "test_top1_accuracy": 0.4304,
        "test_top3_accuracy": 0.7089,
        "test_top5_accuracy": 0.7911,
        "test_macro_f1": 0.3855,
        "notes": "Best WLASL100 model so far. Strong Top-5 and stable improvement over baseline."
    }

result_rows.append(bigru_row)

# Transformer result
if TRANSFORMER_RESULT_FILE.exists():
    transformer_df = pd.read_csv(TRANSFORMER_RESULT_FILE)
    row = transformer_df.iloc[0].to_dict()

    transformer_row = {
        "dataset": row.get("dataset", "WLASL100"),
        "model": row.get("model", "Small Transformer Encoder"),
        "clean_samples": int(row.get("clean_samples", 1013)),
        "classes": int(row.get("classes", 100)),
        "input_shape": row.get("input_shape", "(60, 516)"),
        "velocity_features": bool(row.get("velocity_features", True)),
        "best_val_f1": float(row.get("best_val_f1", 0.3740)),
        "best_val_top5": float(row.get("best_val_top5", 0.7592)),
        "checkpoint_epoch": int(row.get("checkpoint_epoch", 59)),
        "test_top1_accuracy": float(row.get("test_top1_accuracy", 0.3354)),
        "test_top3_accuracy": float(row.get("test_top3_accuracy", 0.6013)),
        "test_top5_accuracy": float(row.get("test_top5_accuracy", 0.6709)),
        "test_macro_f1": float(row.get("test_macro_f1", 0.2830)),
        "notes": "Useful comparison model, but weaker than BiGRU on WLASL100."
    }
else:
    transformer_row = {
        "dataset": "WLASL100",
        "model": "Small Transformer Encoder",
        "clean_samples": 1013,
        "classes": 100,
        "input_shape": "(60, 516)",
        "velocity_features": True,
        "best_val_f1": 0.3740,
        "best_val_top5": 0.7592,
        "checkpoint_epoch": 59,
        "test_top1_accuracy": 0.3354,
        "test_top3_accuracy": 0.6013,
        "test_top5_accuracy": 0.6709,
        "test_macro_f1": 0.2830,
        "notes": "Useful comparison model, but weaker than BiGRU on WLASL100."
    }

result_rows.append(transformer_row)

comparison_df = pd.DataFrame(result_rows)

comparison_df

## 2. Clean comparison table

This table focuses on the metrics that matter most for selecting the next architecture.

In [ ]:
metric_columns = [
    "model",
    "input_shape",
    "velocity_features",
    "best_val_f1",
    "best_val_top5",
    "test_top1_accuracy",
    "test_top3_accuracy",
    "test_top5_accuracy",
    "test_macro_f1",
    "checkpoint_epoch",
    "notes"
]

clean_comparison_df = comparison_df[metric_columns].copy()

clean_comparison_df

## 3. Save comparison results

In [ ]:
comparison_df.to_csv(COMPARISON_FILE, index=False)

print("Saved WLASL100 model comparison to:")
print(COMPARISON_FILE)

## 4. Compare validation performance

Validation Macro F1 is important because it gives a class-balanced view of performance.

In [ ]:
plot_df = comparison_df.dropna(subset=["best_val_f1"]).copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["model"], plot_df["best_val_f1"])
plt.title("WLASL100 Best Validation Macro F1 by Model")
plt.xlabel("Model")
plt.ylabel("Best Validation Macro F1")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = comparison_df.dropna(subset=["best_val_top5"]).copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["model"], plot_df["best_val_top5"])
plt.title("WLASL100 Best Validation Top-5 Accuracy by Model")
plt.xlabel("Model")
plt.ylabel("Best Validation Top-5 Accuracy")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 5. Compare test performance

In [ ]:
test_metric_df = comparison_df.dropna(subset=["test_top1_accuracy", "test_top5_accuracy", "test_macro_f1"]).copy()

test_metric_df[[
    "model",
    "test_top1_accuracy",
    "test_top3_accuracy",
    "test_top5_accuracy",
    "test_macro_f1"
]]

In [ ]:
test_metric_df = comparison_df.dropna(subset=["test_top1_accuracy"]).copy()

metrics_to_plot = [
    "test_top1_accuracy",
    "test_top3_accuracy",
    "test_top5_accuracy",
    "test_macro_f1"
]

x = np.arange(len(test_metric_df["model"]))
width = 0.2

plt.figure(figsize=(12, 6))

for i, metric in enumerate(metrics_to_plot):
    plt.bar(x + i * width, test_metric_df[metric], width, label=metric)

plt.title("WLASL100 Test Performance by Model")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(x + width * 1.5, test_metric_df["model"], rotation=25, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Improvement from baseline to improved BiGRU

This section highlights the impact of the model improvement work.

In [ ]:
baseline_f1 = comparison_df.loc[comparison_df["model"] == "Baseline BiLSTM", "best_val_f1"].values[0]
bigru_f1 = comparison_df.loc[comparison_df["model"] == "BiGRU + Temporal Attention", "best_val_f1"].values[0]

f1_improvement = bigru_f1 - baseline_f1
relative_f1_improvement = bigru_f1 / baseline_f1 if baseline_f1 > 0 else np.nan

print("Baseline validation F1:", round(baseline_f1, 4))
print("Improved BiGRU validation F1:", round(bigru_f1, 4))
print("Absolute F1 improvement:", round(f1_improvement, 4))
print("Relative F1 improvement:", round(relative_f1_improvement, 2), "x")

## 7. BiGRU vs Transformer comparison

This directly compares the two serious model candidates.

In [ ]:
bigru = comparison_df[comparison_df["model"] == "BiGRU + Temporal Attention"].iloc[0]
transformer = comparison_df[comparison_df["model"] == "Small Transformer Encoder"].iloc[0]

direct_comparison = pd.DataFrame([
    {
        "metric": "Best Validation F1",
        "bigru_attention": bigru["best_val_f1"],
        "transformer": transformer["best_val_f1"],
        "winner": "BiGRU + Attention" if bigru["best_val_f1"] > transformer["best_val_f1"] else "Transformer"
    },
    {
        "metric": "Best Validation Top-5",
        "bigru_attention": bigru["best_val_top5"],
        "transformer": transformer["best_val_top5"],
        "winner": "BiGRU + Attention" if bigru["best_val_top5"] > transformer["best_val_top5"] else "Transformer"
    },
    {
        "metric": "Test Top-1 Accuracy",
        "bigru_attention": bigru["test_top1_accuracy"],
        "transformer": transformer["test_top1_accuracy"],
        "winner": "BiGRU + Attention" if bigru["test_top1_accuracy"] > transformer["test_top1_accuracy"] else "Transformer"
    },
    {
        "metric": "Test Top-3 Accuracy",
        "bigru_attention": bigru["test_top3_accuracy"],
        "transformer": transformer["test_top3_accuracy"],
        "winner": "BiGRU + Attention" if bigru["test_top3_accuracy"] > transformer["test_top3_accuracy"] else "Transformer"
    },
    {
        "metric": "Test Top-5 Accuracy",
        "bigru_attention": bigru["test_top5_accuracy"],
        "transformer": transformer["test_top5_accuracy"],
        "winner": "BiGRU + Attention" if bigru["test_top5_accuracy"] > transformer["test_top5_accuracy"] else "Transformer"
    },
    {
        "metric": "Test Macro F1",
        "bigru_attention": bigru["test_macro_f1"],
        "transformer": transformer["test_macro_f1"],
        "winner": "BiGRU + Attention" if bigru["test_macro_f1"] > transformer["test_macro_f1"] else "Transformer"
    }
])

direct_comparison

In [ ]:
DIRECT_COMPARISON_FILE = REPORT_DIR / "wlasl100_bigru_vs_transformer_comparison.csv"
direct_comparison.to_csv(DIRECT_COMPARISON_FILE, index=False)

print("Saved direct comparison to:")
print(DIRECT_COMPARISON_FILE)

## 8. Model selection decision

Based on WLASL100 results, BiGRU + Temporal Attention is selected for scaling to WLASL300.

In [ ]:
selected_model = "BiGRU + Temporal Attention"

decision_text = f"""# WLASL100 Model Selection Decision

## Selected model for WLASL300 scaling

**{selected_model}**

## Reason

The BiGRU + Temporal Attention model outperformed the Small Transformer Encoder on all key WLASL100 metrics.

| Metric | BiGRU + Attention | Transformer |
|---|---:|---:|
| Best Validation F1 | {bigru['best_val_f1']:.4f} | {transformer['best_val_f1']:.4f} |
| Best Validation Top-5 | {bigru['best_val_top5']:.4f} | {transformer['best_val_top5']:.4f} |
| Test Top-1 Accuracy | {bigru['test_top1_accuracy']:.4f} | {transformer['test_top1_accuracy']:.4f} |
| Test Top-3 Accuracy | {bigru['test_top3_accuracy']:.4f} | {transformer['test_top3_accuracy']:.4f} |
| Test Top-5 Accuracy | {bigru['test_top5_accuracy']:.4f} | {transformer['test_top5_accuracy']:.4f} |
| Test Macro F1 | {bigru['test_macro_f1']:.4f} | {transformer['test_macro_f1']:.4f} |

## Interpretation

The Transformer model was useful as a comparison, but it underperformed on WLASL100. This is likely because WLASL100 has only 1,013 clean samples across 100 classes, making it too small for the Transformer to learn robust temporal patterns.

The BiGRU + Temporal Attention model is more suitable for the current dataset size because it learns movement sequences efficiently while still using attention to focus on important frames.

## Next step

Scale the BiGRU + Temporal Attention architecture to WLASL300.

Planned notebooks:

1. `09_prepare_wlasl300_dataset.ipynb`
2. `10_extract_wlasl300_keypoints.ipynb`
3. `11_train_bigru_attention_wlasl300.ipynb`
4. `12_evaluate_wlasl300_model.ipynb`

## Note

The Transformer should not be discarded permanently. It can be tested again later on WLASL1000 or WLASL2000, where there may be enough data for it to become more effective.
"""

DECISION_FILE.write_text(decision_text, encoding="utf-8")

print("Saved model selection decision to:")
print(DECISION_FILE)

print(decision_text)

## 9. Final summary

In [ ]:
print("Final WLASL100 model comparison summary")
print("--------------------------------------")
print("Dataset: WLASL100")
print("Clean samples: 1,013")
print("Classes: 100")
print()
print("Baseline BiLSTM:")
print(f"- Best Validation F1: {baseline_f1:.4f}")
print()
print("Improved BiGRU + Temporal Attention:")
print(f"- Best Validation F1: {bigru['best_val_f1']:.4f}")
print(f"- Best Validation Top-5: {bigru['best_val_top5']:.4f}")
print(f"- Test Top-1 Accuracy: {bigru['test_top1_accuracy']:.4f}")
print(f"- Test Top-3 Accuracy: {bigru['test_top3_accuracy']:.4f}")
print(f"- Test Top-5 Accuracy: {bigru['test_top5_accuracy']:.4f}")
print(f"- Test Macro F1: {bigru['test_macro_f1']:.4f}")
print()
print("Small Transformer Encoder:")
print(f"- Best Validation F1: {transformer['best_val_f1']:.4f}")
print(f"- Best Validation Top-5: {transformer['best_val_top5']:.4f}")
print(f"- Test Top-1 Accuracy: {transformer['test_top1_accuracy']:.4f}")
print(f"- Test Top-3 Accuracy: {transformer['test_top3_accuracy']:.4f}")
print(f"- Test Top-5 Accuracy: {transformer['test_top5_accuracy']:.4f}")
print(f"- Test Macro F1: {transformer['test_macro_f1']:.4f}")
print()
print("Selected model for WLASL300:", selected_model)
print()
print("Saved files:")
print("-", COMPARISON_FILE)
print("-", DIRECT_COMPARISON_FILE)
print("-", DECISION_FILE)